# Exercise 2.2.2 — Data Types and Cleaning

This exercise continues with `datania_households_raw.csv` from Exercise 2.2.1. You will practice:

- Understanding the difference between DataFrame and Series
- Fixing categorical columns with `.replace()` — and why `.str.replace()` is the wrong tool here
- Building a cleaning pipeline for messy numeric data, step by step
- Wrapping cleaning logic in a reusable function and applying it with `.apply()`
- Converting text to dates and using the `.dt` accessor
- Using row-wise `apply(axis=1)` to flag suspicious records
- Saving the cleaned output to `10_cleaned/` and verifying it round-trips correctly

### Path Setup (run first)

In [ ]:
import os
import numpy as np
import pandas as pd

DATA_RAW_DIR = '../../data/0_raw'
FILE_NAME = 'tanzania/datania_households_raw.csv'
raw_path = os.path.join(DATA_RAW_DIR, FILE_NAME)

df = pd.read_csv(raw_path, dtype={'hh_id': str, 'region_code': str})

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
df.head(5)

---

## Task 1 — DataFrame vs Series

Selecting a single column from a DataFrame returns a **Series** — a one-dimensional object with its own dtype and methods. Most type operations (`.dtype`, `.str`, `.dt`, `.value_counts()`) work on Series, not on the full DataFrame.

In [ ]:
print(type(df))
print(type(df['income_dkw']))

In [ ]:
# Print the dtype pandas assigned to each column
df. # your code here

**Questions:**

- What dtype did pandas assign to `income_dkw`? Is it numeric? Why?
- What dtype is `survey_date`? What would you need to do before extracting the month from it?
- `hh_id` and `region_code` are `object`. Is that intentional here? What would have happened without the `dtype` argument in `read_csv`?

---

## Task 2 — Fix categorical columns: `.replace()` vs `.str.replace()`

Two methods look similar but do very different things:

- **`.replace({'old': 'new'})`** — matches and replaces **whole values** (exact match)
- **`.str.replace('old', 'new')`** — finds and replaces a **substring** within each string

This distinction matters. Inspect both columns, then fix them.

In [ ]:
print('urban_rural:', df['urban_rural'].value_counts(dropna=False))
print()
print('region_code:', df['region_code'].value_counts(dropna=False))

In [ ]:
# Fix the typo in urban_rural: 'Urbn' → 'Urban'
df['urban_rural'] = df['urban_rural']. # your code here — use .replace()

print(df['urban_rural'].value_counts(dropna=False))

In [ ]:
# region_code has two problems:
#   '99' is a coded missing value — replace with NaN
#   '1'  should be '01'           — normalise to 2-digit format
#
# ⚠️ Why NOT use .str.replace('1', '01')?
# It replaces every '1' character inside the string:
#   '01' → '001', '21' → '021', '16' → '016'  — all wrong.
# .replace() with a dict does exact-value matching — only '1' is changed.

df['region_code'] = df['region_code'].replace( # your code here — replace '99' with np.nan )
df['region_code'] = df['region_code'].replace( # your code here — replace '1' with '01' )

print(df['region_code'].value_counts(dropna=False))

**Questions:**

- Why would `.str.replace('1', '01')` break codes like `'01'`, `'21'`, or `'16'`?
- After your fix, how many `NaN` values are there in `region_code`? What caused each one?
- When would `.str.replace()` be the right tool? Give an example.

---

## Task 3 — Clean `income_dkw`: from messy text to a numeric column

The `income_dkw` column holds household income but the raw values contain spaces, commas, the prefix `Ar`, text labels like `unknown`, and negative values. We will clean it in four steps.

**Step 1: Inspect before writing any code.**

In [ ]:
# What distinct values exist in this column?
df['income_dkw']. # your code here

**Step 2: Build the cleaning chain interactively.**

Chain `.str` operations one at a time so you can see what each step does. Work on a copy so the original column stays intact for comparison.

In [ ]:
income_text = df['income_dkw'].astype('string')

income_text = income_text.str.replace(' ', '', regex=False)   # remove spaces
income_text = income_text.str.replace('Ar', '', regex=False)  # remove currency prefix
income_text = income_text.str. # your code here — remove commas (thousands separator)
income_text = income_text.replace('unknown', np.nan)          # whole-value replacement
income_text = income_text.replace( # your code here — replace 'NA' with np.nan )

income_text.unique()

**Step 3: Collapse into a single chain and store as a new column.**

In [ ]:
df['income_dkw_clean'] = (
    df['income_dkw']
    .astype('string')
    # your code here — chain all the cleaning steps from Step 2
)

df[['income_dkw', 'income_dkw_clean']].head(15)

**Step 4: Wrap the logic in a reusable function and apply it.**

Functions make cleaning logic testable and reusable across datasets. Complete the function body, test it on individual values, then apply it to the whole column with `.apply()`.

In [ ]:
def clean_income(value, chars_to_remove=None, null_codes=None):
    """Clean a messy income string: remove formatting and replace invalid codes with NaN."""
    if chars_to_remove is None:
        chars_to_remove = [' ', 'Ar', ',']
    if null_codes is None:
        null_codes = ['unknown', 'NA', 'nan', 'not recorded']

    text = str(value)                       # handles NaN → 'nan', int → '1200', etc.
    for char in chars_to_remove:
        # your code here — remove each character from text
        pass
    if # your code here — check whether text is in null_codes:
        return np.nan
    return text


# Test on individual values before applying to the whole column
print(clean_income('Ar 32,000'))    # → '32000'
print(clean_income('45 000'))       # → '45000'
print(clean_income('unknown'))      # → nan
print(clean_income(np.nan))         # → nan

In [ ]:
# Apply the function to the entire column
# .apply() calls clean_income once for every value in the Series
df['income_dkw_clean'] = df['income_dkw']. # your code here

# Convert the cleaned text to float — errors='coerce' turns unparseable values into NaN
df['income_dkw_num'] = pd.to_numeric( # your code here )

df[['income_dkw', 'income_dkw_clean', 'income_dkw_num']].head(15)

In [ ]:
# Which rows still have NaN after conversion?
df[df['income_dkw_num'].isna()][['hh_id', 'income_dkw', 'income_dkw_clean']]

**Questions:**

- How many values are still `NaN` after conversion? What is the raw value for each?
- What is the difference between `.str.replace()` and `.replace()` in Step 2? Which one replaces substrings and which matches whole values?
- Why test the function on individual values *before* calling `.apply()`?

---

## Task 4 — Convert `survey_date` to datetime

Date columns stored as text prevent any date-based analysis. Use `pd.to_datetime()` to convert, then use the `.dt` accessor to extract date parts.

In [ ]:
df['survey_date'].unique()

In [ ]:
# Convert — coerce invalid values to NaT
df['survey_date_parsed'] = pd.to_datetime( # your code here )

# Which rows failed to parse?
df[df['survey_date_parsed'].isna()][['hh_id', 'survey_date']]

In [ ]:
# Extract month using the .dt accessor
df['survey_month'] = df['survey_date_parsed'].dt. # your code here
df['survey_month'].value_counts(dropna=False).sort_index()

**Questions:**

- Which raw date values failed to parse? What is wrong with each one?
- What is `NaT`? How is it different from `NaN`?
- What other `.dt` properties could you extract? Try `.dt.day_name()` on the parsed column.

---

## Task 5 — Row-wise `apply(axis=1)`

So far, `.apply()` called a function once per **cell** on a single column. With `apply(axis=1)`, the function receives an entire **row** as a Series. This lets you combine values from multiple columns in a single operation.

First, compute income per capita — a value that depends on two columns.

In [ ]:
def compute_income_per_capita(row):
    """Return income per household member, or NaN if inputs are invalid."""
    income  = row['income_dkw_num']
    hh_size = row['hh_size']

    if # your code here — check if income is missing:
        return np.nan
    if # your code here — check if hh_size is missing or <= 0:
        return np.nan
    return round(income / hh_size, 2)


# axis=1 → call the function once per row
df['income_per_capita'] = df. # your code here

df[['hh_id', 'hh_size', 'income_dkw_num', 'income_per_capita']].head(10)

Now flag suspicious rows — records that have either an impossible `hh_size` or an implausibly high income. Use a row-wise lambda.

In [ ]:
# Flag rows where income > 500 000 (implausible) OR hh_size <= 0 (impossible).
#
# pd.notna() guards against comparing NaN with >, which would silently return False.

df['suspicious'] = df.apply(
    lambda row: (
        # your code here — condition A: income is not missing AND > 500_000
    ) or (
        # your code here — condition B: hh_size <= 0
    ),
    axis=1
)

print(f"{df['suspicious'].sum()} suspicious rows found:")
df[df['suspicious']][['hh_id', 'hh_size', 'income_dkw', 'income_dkw_num']]

**Questions:**

- Why do we use `pd.notna(row['income_dkw_num'])` before checking `> 500_000`? What would happen if we skipped it?
- `compute_income_per_capita` is a named function; the flag uses a lambda. When would you prefer one over the other?
- A flagged row is not yet removed. Why flag first rather than drop immediately?

---

## Task 6 — Save to `10_cleaned/`

Raw data lives in `0_raw/` and must never be modified. Cleaned outputs go to `10_cleaned/` — a separate folder that makes the transformation explicit and reproducible.

Before saving, decide which columns belong in the cleaned file. It should include identifiers, cleaned column versions, and derived columns worth keeping. It should **not** include raw messy columns alongside their cleaned counterparts — those belong in a cleaning log, not the output.

In [ ]:
# Define which columns to keep in the cleaned output
# Think: identifiers, cleaned values, derived columns — drop raw intermediates
COLS_OUT = # your code here — a list of column names

df_clean = df[COLS_OUT].copy()
print('Shape:', df_clean.shape)
df_clean.head()

In [ ]:
DATA_CLEAN_DIR = '../../data/10_cleaned'
os.makedirs(DATA_CLEAN_DIR, exist_ok=True)

output_path = os.path.join(DATA_CLEAN_DIR, 'datania_households_clean.csv')

# Save — why index=False?
df_clean.to_csv( # your code here )

print('Saved to:', output_path)

In [ ]:
# Reload and spot-check — does the file round-trip correctly?
df_check = pd.read_csv(output_path)
print('Reloaded shape:', df_check.shape)
print()
print(df_check.dtypes)
df_check.head()

**Questions:**

- What happens if you omit `index=False`? Try it and reload the file to see.
- After reloading, what dtype does `survey_date_parsed` have? What does that tell you about what CSV cannot preserve?
- Should the `suspicious` column be in the cleaned output, excluded, or handled differently before saving? Why?